# 01 — Input rasters (fabric-bounded)

Snapshots of every CONUS/shared and zonal-source raster that feeds the parameterization, clipped to the active fabric's bounds with the HRU outline overlaid. Missing sources (e.g. an unused LULC source) are skipped with a warning.

> **Run this in JupyterHub on a compute node with enough `--mem`.** Rendering a
> full fabric (especially CONUS `gfv2`, ~361k HRUs) loads large geometries and
> rasters; the login node will run out of memory. Pick the fabric with the
> `FABRIC` env var (or edit the first code cell). Set `SAVE_FIGURES=1` (or
> `viz.SAVE_FIGURES = True`) to write PNGs into `docs/figures/{FABRIC}/`, or
> batch-generate them with `python scripts/render_figures.py --fabric {FABRIC}`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt

from gfv2_params import viz
from gfv2_params.config import load_base_config

# Fabric is selected by the FABRIC env var (set it in your JupyterHub session),
# or edit the default here. SAVE_FIGURES=1 writes PNGs to docs/figures/{FABRIC}/.
FABRIC = os.environ.get("FABRIC", "oregon")
viz.FABRIC = FABRIC
viz.SAVE_FIGURES = os.environ.get("SAVE_FIGURES", "0") == "1"

cfg = load_base_config(fabric=FABRIC)
DATA_ROOT = Path(cfg["data_root"])
HRU_GPKG = Path(cfg["hru_gpkg"])
HRU_LAYER = cfg["hru_layer"]
ID_FEATURE = cfg["id_feature"]

print(f"FABRIC       = {FABRIC}")
print(f"SAVE_FIGURES = {viz.SAVE_FIGURES}  ->  docs/figures/{FABRIC}/")
print(f"data_root    = {DATA_ROOT}")
print(f"hru_gpkg     = {HRU_GPKG} (layer={HRU_LAYER}, id={ID_FEATURE})")

from gfv2_params.config import load_config
import gfv2_params.config as _cfgmod
ZONAL_YML = Path(_cfgmod.__file__).resolve().parents[2] / "configs" / "zonal" / "zonal_params.yml"
zonal_cfg = load_config(ZONAL_YML, fabric=FABRIC)

## Build the raster inventory

Shared hydrology/DEM VRTs (from the fabric profile + `shared/conus/vrt`) plus the zonal `source_raster` entries from `zonal_params.yml`.

In [ ]:
inventory = viz.dedupe_raster_entries(
    viz.shared_raster_inventory(cfg) + viz.zonal_source_inventory(zonal_cfg)
)
for e in inventory:
    print(f"{e.name:16} {e.kind:11} {e.path}")

## Read the fabric outline once

Dissolved boundary, reused (reprojected per raster CRS) as an overlay. Fabric bounds are cached per CRS so the geometry is read only once.

In [ ]:
import geopandas as gpd

fabric_geom = gpd.read_file(HRU_GPKG, layer=HRU_LAYER)
fabric_geom = fabric_geom[fabric_geom.geometry.notna() & ~fabric_geom.geometry.is_empty]
outline = fabric_geom.dissolve().boundary

_bounds_cache = {}
def bounds_for(crs):
    key = crs.to_string() if crs is not None else None
    if key not in _bounds_cache:
        _bounds_cache[key] = tuple(fabric_geom.to_crs(crs).total_bounds)
    return _bounds_cache[key]

## Plot each raster clipped to the fabric

In [ ]:
import rasterio

for entry in inventory:
    with rasterio.open(entry.path) as src:
        rcrs = src.crs
    b = bounds_for(rcrs)
    arr, extent = viz.clip_overview(entry.path, b)
    fig, ax = plt.subplots(figsize=(8, 8))
    viz.plot_raster(ax, arr, extent=extent, cmap=entry.cmap,
                    categorical=(entry.kind == "categorical"),
                    title=f"{entry.name}  [{FABRIC}]", label=entry.units)
    try:
        outline.to_crs(rcrs).plot(ax=ax, color="red", linewidth=0.6)
    except Exception as exc:
        print(f"  (outline overlay skipped for {entry.name}: {exc})")
    viz.save_figure(fig, f"input_raster_{entry.name}")
    plt.show()